# Physical-pruning structure and inference-graph diagnostics

This notebook compares a baseline HiFi-GAN generator against its physically pruned counterpart. It prints per-layer tensor/channel shapes, parameter counts, MAC estimates, flags SIMD-unfriendly channel dimensions, and optionally exports/inspects ONNX graphs.

The defaults target the checked-in 512C baseline and 70% physical-pruning checkpoints. Change the four paths below to inspect 256C or 128C models.

In [29]:
from pathlib import Path
import copy
import json
import sys
import warnings

import pandas as pd
import torch
import torch.nn as nn

ROOT = Path.cwd()
if not (ROOT / 'models.py').exists():
    raise RuntimeError('Run this notebook with hifi-gan as the working directory.')
sys.path.insert(0, str(ROOT))

BASELINE_CONFIG = ROOT / 'config_v1_128.json'
BASELINE_CKPT = ROOT / 'cp_hifigan/v1_c128/g_07960000'
PRUNED_CONFIG = ROOT / 'config_v1_128_p70.json'
PRUNED_CKPT = ROOT / 'cp_hifigan/v1_c128_physical70/g_00265000'

# The diagnostic input represents one 80-bin mel spectrogram.
MEL_FRAMES = 100
ALIGNMENTS = (8, 16, 32)  # useful CPU SIMD / accelerator tiling checks
DEVICE = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', DEVICE)

PyTorch: 2.4.1+cu121 | device: cpu


In [30]:
from env import AttrDict
from models import Generator
from inference import prepare_generator_for_checkpoint

def load_generator(config_path, checkpoint_path):
    """Load plain, weight-normalized, or mask-pruned project checkpoints."""
    with open(config_path) as f:
        h = AttrDict(json.load(f))
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    state_dict = checkpoint['generator'] if 'generator' in checkpoint else checkpoint
    model = Generator(h).to(DEVICE)
    checkpoint_kind = prepare_generator_for_checkpoint(model, state_dict)
    model.eval()
    return model, h, checkpoint_kind

baseline, baseline_h, baseline_kind = load_generator(BASELINE_CONFIG, BASELINE_CKPT)
pruned, pruned_h, pruned_kind = load_generator(PRUNED_CONFIG, PRUNED_CKPT)
print(f'baseline: {baseline_kind}; pruning ratio={baseline_h.get("physical_prune_ratio", 0)}')
print(f'pruned:   {pruned_kind}; pruning ratio={pruned_h.physical_prune_ratio}')

/home/woody/vlbi/vlbi107v/miniconda3/envs/hifi-gan/lib/python3.8/site-packages/torch/nn/utils/weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Checkpoint contains weight_norm parameters. Keeping weight_norm wrappers for loading...
Checkpoint contains plain conv weights. Removing weight_norm wrappers before loading...
Removing weight norm...
baseline: weight_norm; pruning ratio=0
pruned:   plain; pruning ratio=0.7


In [31]:
def _as_shape(x):
    return tuple(x.shape) if isinstance(x, torch.Tensor) else None


def _macs(module, input_tensor, output_tensor):
    # One multiply-accumulate is counted as one MAC (not two FLOPs).
    if not isinstance(module, (nn.Conv1d, nn.ConvTranspose1d)):
        return 0
    batch = output_tensor.shape[0]
    output_positions = output_tensor.shape[-1]
    kernel = module.kernel_size[0]
    return int(batch * output_positions * module.out_channels *
               (module.in_channels // module.groups) * kernel)


def inspect_layers(model, mel_frames=MEL_FRAMES):
    rows, hooks = [], []

    def hook(name):
        def record(module, inputs, output):
            x = inputs[0]
            if not isinstance(x, torch.Tensor) or not isinstance(output, torch.Tensor):
                return
            in_c = getattr(module, 'in_channels', x.shape[1])
            out_c = getattr(module, 'out_channels', output.shape[1])
            channels = (int(in_c), int(out_c))
            irregular = [a for a in ALIGNMENTS if any(c % a for c in channels)]
            rows.append({
                'layer': name, 'type': type(module).__name__,
                'input_shape': _as_shape(x), 'output_shape': _as_shape(output),
                'in_channels': int(in_c), 'out_channels': int(out_c),
                'kernel': tuple(module.kernel_size), 'groups': int(module.groups),
                'parameters': sum(p.numel() for p in module.parameters(recurse=False)),
                'MACs': _macs(module, x, output),
                'misaligned_to': ','.join(map(str, irregular)) or 'none',
            })
        return record

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv1d, nn.ConvTranspose1d)):
            hooks.append(module.register_forward_hook(hook(name)))
    with torch.inference_mode():
        output = model(torch.randn(1, 80, mel_frames, device=DEVICE))
    for h in hooks:
        h.remove()
    return pd.DataFrame(rows), output


baseline_layers, baseline_output = inspect_layers(baseline)
pruned_layers, pruned_output = inspect_layers(pruned)
print('Generator output tensors:')
print('baseline_output:', baseline_output)
print('pruned_output:', pruned_output)
print('Generator output shapes:', tuple(baseline_output.shape), tuple(pruned_output.shape))


def print_channel_counts(label, layers):
    print(f'\n{label} channel counts:')
    for column in ['in_channels', 'out_channels']:
        counts = layers[column].astype(int).value_counts().sort_index()
        print(f'  {column}:')
        print(counts.to_string())


print_channel_counts('Baseline', baseline_layers)
print_channel_counts('Physical-pruned', pruned_layers)


def print_summary(label, layers):
    print(f'\n{label}: {len(layers)} convolution operators')
    print(f"  conv parameters: {layers['parameters'].sum():,}")
    print(f"  estimated MACs / {MEL_FRAMES} mel frames: {layers['MACs'].sum():,}")
    print(f"  estimated FLOPs (2 × MACs): {2 * layers['MACs'].sum():,}")
    print(f"  irregular operators: {(layers['misaligned_to'] != 'none').sum()} / {len(layers)}")


print_summary('Baseline', baseline_layers)
print_summary('Physical-pruned', pruned_layers)

Generator output tensors:
baseline_output: tensor([[[-0.2572, -0.0375,  0.0098,  ..., -0.1650, -0.0408, -0.0718]]])
pruned_output: tensor([[[-0.3532, -0.2217, -0.2061,  ...,  0.3244,  0.3978,  0.4784]]])
Generator output shapes: (1, 1, 25600) (1, 1, 25600)

Baseline channel counts:
  in_channels:
in_channels
8      19
16     19
32     19
64     19
80      1
128     1
  out_channels:
out_channels
1       1
8      19
16     19
32     19
64     19
128     1

Physical-pruned channel counts:
  in_channels:
in_channels
2       9
5       9
8      10
10      9
16     10
19      9
32     10
64     10
80      1
128     1
  out_channels:
out_channels
1       1
2       9
5       9
8      10
10      9
16     10
19      9
32     10
64     10
128     1

Baseline: 78 convolution operators
  conv parameters: 928,514
  estimated MACs / 100 mel frames: 2,220,441,600
  estimated FLOPs (2 × MACs): 4,440,883,200
  irregular operators: 40 / 78

Physical-pruned: 78 convolution operators
  conv parameters: 445

In [24]:
# One row per convolution: this is the requested before/after layer-shape report.
compare = baseline_layers.merge(
    pruned_layers, on='layer', how='outer', suffixes=('_baseline', '_pruned')
).sort_values('layer').reset_index(drop=True)
compare['parameter_reduction_%'] = (
    100 * (1 - compare['parameters_pruned'] / compare['parameters_baseline'])
).round(2)
compare['MAC_reduction_%'] = (
    100 * (1 - compare['MACs_pruned'] / compare['MACs_baseline'])
).round(2)
columns = ['layer', 'type_baseline', 'input_shape_baseline', 'output_shape_baseline',
           'input_shape_pruned', 'output_shape_pruned', 'in_channels_baseline',
           'out_channels_baseline', 'in_channels_pruned', 'out_channels_pruned',
           'parameters_baseline', 'parameters_pruned', 'parameter_reduction_%',
           'MACs_baseline', 'MACs_pruned', 'MAC_reduction_%', 'misaligned_to_pruned']
display(compare[columns])

# Saving makes the evidence easy to reference in a paper or profiling report.
compare.to_csv(ROOT / 'physical_pruning_layer_comparison.csv', index=False)
print('Saved physical_pruning_layer_comparison.csv')

,layer,type_baseline,input_shape_baseline,output_shape_baseline,input_shape_pruned,output_shape_pruned,in_channels_baseline,out_channels_baseline,in_channels_pruned,out_channels_pruned,parameters_baseline,parameters_pruned,parameter_reduction_%,MACs_baseline,MACs_pruned,MAC_reduction_%,misaligned_to_pruned
0,conv_post,Conv1d,"(1, 32, 25600)","(1, 1, 25600)","(1, 32, 25600)","(1, 1, 25600)",32,1,32,1,226,225,0.44,5734400,5734400,0.00,"8,16,32"
1,conv_pre,Conv1d,"(1, 80, 100)","(1, 512, 100)","(1, 80, 100)","(1, 512, 100)",80,512,80,512,287744,287232,0.18,28672000,28672000,0.00,32
2,resblocks.0.convs1.0,Conv1d,"(1, 256, 800)","(1, 256, 800)","(1, 256, 800)","(1, 77, 800)",256,256,256,77,197120,59213,69.96,157286400,47308800,69.92,"8,16,32"
3,resblocks.0.convs1.1,Conv1d,"(1, 256, 800)","(1, 256, 800)","(1, 256, 800)","(1, 77, 800)",256,256,256,77,197120,59213,69.96,157286400,47308800,69.92,"8,16,32"
4,resblocks.0.convs1.2,Conv1d,"(1, 256, 800)","(1, 256, 800)","(1, 256, 800)","(1, 77, 800)",256,256,256,77,197120,59213,69.96,157286400,47308800,69.92,"8,16,32"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,resblocks.9.convs2.2,Conv1d,"(1, 32, 25600)","(1, 32, 25600)","(1, 10, 25600)","(1, 32, 25600)",32,32,10,32,3136,992,68.37,78643200,24576000,68.75,"8,16,32"
74,ups.0,ConvTranspose1d,"(1, 512, 100)","(1, 256, 800)","(1, 512, 100)","(1, 256, 800)",512,256,512,256,2097920,2097408,0.02,1677721600,1677721600,0.00,none
75,ups.1,ConvTranspose1d,"(1, 256, 800)","(1, 128, 6400)","(1, 256, 800)","(1, 128, 6400)",256,128,256,128,524672,524416,0.05,3355443200,3355443200,0.00,none
76,ups.2,ConvTranspose1d,"(1, 128, 6400)","(1, 64, 12800)","(1, 128, 6400)","(1, 64, 12800)",128,64,128,64,32960,32832,0.39,419430400,419430400,0.00,none


Saved physical_pruning_layer_comparison.csv


In [25]:
# Focus on changed layers and inspect remaining channel widths.
changed = compare.query('in_channels_baseline != in_channels_pruned or out_channels_baseline != out_channels_pruned')
print('Changed convolution layers:', len(changed), 'of', len(compare))
display(changed[['layer', 'in_channels_baseline', 'out_channels_baseline',
                 'in_channels_pruned', 'out_channels_pruned',
                 'parameter_reduction_%', 'MAC_reduction_%', 'misaligned_to_pruned']])

unchanged_upsampling = compare[compare['layer'].str.startswith('ups.')][
    ['layer', 'in_channels_pruned', 'out_channels_pruned', 'MACs_pruned']
]
print('\nUpsampling layers remain structurally unchanged by this implementation:')
display(unchanged_upsampling)

pruned_irregular = pruned_layers.query("misaligned_to != 'none'")
print('\nPhysical pruning produces these potentially irregular Conv dimensions:')
display(pruned_irregular[['layer', 'type', 'in_channels', 'out_channels', 'misaligned_to']])

Changed convolution layers: 72 of 78


,layer,in_channels_baseline,out_channels_baseline,in_channels_pruned,out_channels_pruned,parameter_reduction_%,MAC_reduction_%,misaligned_to_pruned
2,resblocks.0.convs1.0,256,256,256,77,69.96,69.92,"8,16,32"
3,resblocks.0.convs1.1,256,256,256,77,69.96,69.92,"8,16,32"
4,resblocks.0.convs1.2,256,256,256,77,69.96,69.92,"8,16,32"
5,resblocks.0.convs2.0,256,256,77,256,69.87,69.92,"8,16,32"
6,resblocks.0.convs2.1,256,256,77,256,69.87,69.92,"8,16,32"
...,...,...,...,...,...,...,...,...
69,resblocks.9.convs1.1,32,32,32,10,69.07,68.75,"8,16,32"
70,resblocks.9.convs1.2,32,32,32,10,69.07,68.75,"8,16,32"
71,resblocks.9.convs2.0,32,32,10,32,68.37,68.75,"8,16,32"
72,resblocks.9.convs2.1,32,32,10,32,68.37,68.75,"8,16,32"



Upsampling layers remain structurally unchanged by this implementation:


,layer,in_channels_pruned,out_channels_pruned,MACs_pruned
74,ups.0,512,256,1677721600
75,ups.1,256,128,3355443200
76,ups.2,128,64,419430400
77,ups.3,64,32,209715200



Physical pruning produces these potentially irregular Conv dimensions:


,layer,type,in_channels,out_channels,misaligned_to
0,conv_pre,Conv1d,80,512,32
2,resblocks.0.convs1.0,Conv1d,256,77,"8,16,32"
3,resblocks.0.convs2.0,Conv1d,77,256,"8,16,32"
4,resblocks.0.convs1.1,Conv1d,256,77,"8,16,32"
5,resblocks.0.convs2.1,Conv1d,77,256,"8,16,32"
...,...,...,...,...,...
73,resblocks.11.convs1.1,Conv1d,32,10,"8,16,32"
74,resblocks.11.convs2.1,Conv1d,10,32,"8,16,32"
75,resblocks.11.convs1.2,Conv1d,32,10,"8,16,32"
76,resblocks.11.convs2.2,Conv1d,10,32,"8,16,32"


In [ ]:
# Optional ONNX export and graph inspection. It is skipped cleanly if onnx is unavailable.
def export_onnx(model, name, mel_frames=MEL_FRAMES):
    path = ROOT / f'{name}.onnx'
    example = torch.randn(1, 80, mel_frames, device=DEVICE)
    torch.onnx.export(
        model, example, path, opset_version=17,
        input_names=['mel'], output_names=['audio'],
        dynamic_axes={'mel': {2: 'mel_frames'}, 'audio': {2: 'audio_samples'}},
        do_constant_folding=True, dynamo=False,
    )
    return path

try:
    import onnx
except ImportError:
    print('ONNX package is not installed. Install it with: pip install onnx')
else:
    baseline_onnx = export_onnx(copy.deepcopy(baseline).cpu(), 'baseline_512c')
    pruned_onnx = export_onnx(copy.deepcopy(pruned).cpu(), 'physical_pruned_512c_p70')
    print('Exported:', baseline_onnx.name, 'and', pruned_onnx.name)

    def inspect_onnx(path):
        graph = onnx.shape_inference.infer_shapes(onnx.load(path)).graph
        value_info = {v.name: v for v in list(graph.input) + list(graph.value_info) + list(graph.output)}
        def shape(name):
            v = value_info.get(name)
            if v is None or not v.type.tensor_type.HasField('shape'):
                return None
            return [d.dim_value if d.HasField('dim_value') else d.dim_param
                    for d in v.type.tensor_type.shape.dim]
        rows = []
        for node in graph.node:
            if node.op_type in ('Conv', 'ConvTranspose'):
                attrs = {a.name: list(a.ints) if a.ints else a.i for a in node.attribute}
                rows.append({'op': node.op_type, 'name': node.name,
                             'input': node.input[0], 'input_shape': shape(node.input[0]),
                             'output': node.output[0], 'output_shape': shape(node.output[0]),
                             'attributes': attrs})
        return pd.DataFrame(rows), graph

    baseline_graph, baseline_proto = inspect_onnx(baseline_onnx)
    pruned_graph, pruned_proto = inspect_onnx(pruned_onnx)
    print('ONNX nodes — baseline:', len(baseline_proto.node), '| pruned:', len(pruned_proto.node))
    display(pruned_graph)
    pruned_graph.to_csv(ROOT / 'physical_pruned_512c_p70_onnx_convs.csv', index=False)